# Feature Engineering Post-processing — Nonlinear Only

Input: the output of `feature_engineering.ipynb` / `feature_engineering_utils.py`.

Output: `postprocessed_df` with the **same columns, column order, index, and row order** as the input.

Each step has the same schema so it can be added/removed easily:

```text
previous DataFrame -> one transformation -> next DataFrame
```

Current stages:

```text
feature_engineered_df
    -> log1p positive long-tail features
    -> TRAIN-scaled asinh signed long-tail features
    -> log volatility features
    -> postprocessed_df
```

There is **no winsorization, no mean/std normalization, no z-score clipping, and no missing-value filling** in this notebook. The existing training pipeline remains responsible for train-only mean/std normalization.


In [ ]:
# Step 0A — imports
import gc
from pathlib import Path

import pandas as pd

from feature_engineering_postprocessing_utils import (
    infer_target_cols,
    infer_model_feature_cols,
    infer_feature_groups,
    make_time_train_mask,
    apply_log1p_transform,
    fit_asinh_scales,
    apply_asinh_transform,
    apply_log_volatility_transform,
    save_postprocessing_state,
)


## Step 0B — input and configuration

Targets generated by the feature-engineering utility are detected automatically, including configurable horizons such as `mid_return_t_plus_30`, `target_log_return_30m`, or their `60m` versions.


In [ ]:
# Input DataFrame
if "feature_engineered_df" not in globals():
    if "modeling_df" not in globals():
        raise NameError("Provide feature_engineered_df (or modeling_df) before running this notebook.")
    feature_engineered_df = modeling_df

# TRAIN-only fit range for the signed asinh scale.
TRAIN_START = "2025-01-01"
TRAIN_END   = "2025-07-01"  # [start, end)
TIME_COL = "bucketEnd"

ID_COLS = ["RIC", "date_dt", "bucketEnd"]
TARGET_COLS = infer_target_cols(feature_engineered_df)
EXTRA_EXCLUDE_COLS = [c for c in ["midChange"] if c in feature_engineered_df.columns]

MAX_SAMPLES_PER_FEATURE = 2_000_000
RANDOM_STATE = 42
VOLATILITY_EPSILON = 1e-8
POSTPROCESSING_STATE_PATH = "postprocessing_state.json"

FEATURE_COLS = infer_model_feature_cols(
    feature_engineered_df,
    id_cols=ID_COLS,
    target_cols=TARGET_COLS,
    extra_exclude_cols=EXTRA_EXCLUDE_COLS,
)

train_mask = make_time_train_mask(
    feature_engineered_df,
    time_col=TIME_COL,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
)

print(f"rows: {len(feature_engineered_df):,}")
print(f"columns: {feature_engineered_df.shape[1]}")
print(f"model feature columns: {len(FEATURE_COLS)}")
print("protected target columns:", TARGET_COLS)
print(f"TRAIN rows for asinh-scale fitting: {train_mask.sum():,}")


## Step 1 — classify feature columns

Input: `feature_engineered_df` metadata. Output: `groups`.

Automatic groups can be overridden without editing the utility.


In [ ]:
POSITIVE_LONG_TAIL_ADD = []
SIGNED_LONG_TAIL_ADD = []
VOLATILITY_ADD = []
PASSTHROUGH_ADD = []
BINARY_ADD = []

groups = infer_feature_groups(
    FEATURE_COLS,
    positive_long_tail_add=POSITIVE_LONG_TAIL_ADD,
    signed_long_tail_add=SIGNED_LONG_TAIL_ADD,
    volatility_add=VOLATILITY_ADD,
    passthrough_add=PASSTHROUGH_ADD,
    binary_add=BINARY_ADD,
)

summary = pd.DataFrame({
    "group": ["positive_long_tail", "signed_long_tail", "volatility", "binary", "passthrough"],
    "n_features": [
        len(groups.positive_long_tail),
        len(groups.signed_long_tail),
        len(groups.volatility),
        len(groups.binary),
        len(groups.passthrough),
    ],
})
display(summary)


## Step 2 — positive long-tail: `log1p`

Input: `feature_engineered_df`  
Output: `step2_log1p_df`

For configured non-negative heavy-tail features: `x -> log(1+x)`.


In [ ]:
step2_log1p_df = apply_log1p_transform(
    feature_engineered_df,
    groups.positive_long_tail,
    copy=True,
    negative_policy="nan",
)

print("Step 2 output:", step2_log1p_df.shape)


## Step 3 — signed long-tail: TRAIN-scaled `asinh`

Input: `step2_log1p_df`  
Output: `step3_asinh_df`

Fit `s = median(abs(x_train))` on TRAIN only, then apply `asinh(x/s)` to every row.


In [ ]:
if groups.signed_long_tail:
    asinh_scales = fit_asinh_scales(
        step2_log1p_df,
        groups.signed_long_tail,
        train_mask,
        max_samples_per_feature=MAX_SAMPLES_PER_FEATURE,
        random_state=RANDOM_STATE,
    )
else:
    asinh_scales = {}

step3_asinh_df = apply_asinh_transform(
    step2_log1p_df,
    groups.signed_long_tail,
    asinh_scales,
    copy=True,
)

del step2_log1p_df
gc.collect()

print("Step 3 output:", step3_asinh_df.shape)


## Step 4 — volatility: log transform

Input: `step3_asinh_df`  
Output: `postprocessed_df`

For volatility features: `vol -> log(vol + epsilon)`. No normalization is performed here.


In [ ]:
postprocessed_df = apply_log_volatility_transform(
    step3_asinh_df,
    groups.volatility,
    epsilon=VOLATILITY_EPSILON,
    copy=True,
    negative_policy="nan",
)

del step3_asinh_df
gc.collect()

print("Final postprocessed DataFrame:", postprocessed_df.shape)


## Step 5 — schema and target-protection checks

The final output must have exactly the original schema, and identifiers/targets must be unchanged.


In [ ]:
assert postprocessed_df.shape == feature_engineered_df.shape
assert list(postprocessed_df.columns) == list(feature_engineered_df.columns)
assert postprocessed_df.index.equals(feature_engineered_df.index)

for c in ID_COLS + TARGET_COLS:
    if c in feature_engineered_df.columns:
        assert postprocessed_df[c].equals(feature_engineered_df[c]), f"Unexpected change in protected column: {c}"

print("Schema check: PASS")
print("Protected ID/target columns unchanged: PASS")


## Step 6 — save nonlinear-transform state

Only parameters needed by these nonlinear transforms are saved: feature groups, TRAIN-fitted asinh scales, and volatility epsilon.


In [ ]:
state = save_postprocessing_state(
    POSTPROCESSING_STATE_PATH,
    feature_groups=groups,
    feature_cols=FEATURE_COLS,
    asinh_scales=asinh_scales,
    volatility_epsilon=VOLATILITY_EPSILON,
)

print("Saved:", Path(POSTPROCESSING_STATE_PATH).resolve())


## Final output

Use `postprocessed_df` as input to the existing model-training data preparation. The training pipeline should still run its original **TRAIN-only mean/std normalization once**.
